# 高级编程：SIMD 与 SIMT 混合编程

## 概述

前面我们分别学习了 SIMD 编程模型和 SIMT 编程模型。为了在**复杂控制流、离散访存**等场景下提升向量计算的编程灵活性，在 **Ascend 950PR / Ascend 950DT** 的 AI Vector 核中新增了 SIMT 硬件单元，与原有的 SIMD 计算单元协同工作，形成 **SIMD 与 SIMT 混合编程**模型。本节介绍这一混合编程模型的设计思路和适用场景。

### 前置要求

- 已学习 3.3 SIMD 编程模型，理解多核 SPMD 分片和单核批量向量计算。
- 已学习 3.4 SIMT 编程模型，理解 Grid-Block-Thread 线程层级和线程并发执行模型。
- 本小节为理论讲解，不依赖在线硬件环境。

### 学习目标

学完本小节后，你应该能够：

- 说明为什么在 SIMD 之外还需要引入 SIMT 硬件单元。
- 理解"以 SIMD 为主、SIMT 为辅"的混合编程设计原则。
- 判断哪些场景适合使用 SIMD 与 SIMT 混合编程。

### 小节内容

- 混合编程的设计动机
- 混合编程架构与算力分工
- 混合编程的适用场景

### 混合编程的设计动机与架构

为提升向量计算在**复杂控制流、离散访存**等场景下的编程灵活性，在 **Ascend 950PR / Ascend 950DT** 的 AI Vector 核中新增了 SIMT 硬件单元。在 SIMD 与 SIMT 深度融合的混合编程架构中，矩阵计算单元延续 SIMD 设计，向量计算单元则在 SIMD 基础上引入 SIMT 能力。这一设计确立了**以 SIMD 为主、SIMT 为辅**的混合编程模型，硬件架构图如下：

![](images/03_05_hybrid/hybrid_hardware_architecture.png)

在该模型下，矩阵计算单元和向量计算中的 SIMD 计算单元共同提供**超过 90% 的算力**，为密集计算提供高性能与高算力利用率；向量计算中的 SIMT 则作为**灵活性补充**，专门应对**复杂控制流、离散访存等不规则场景**，提升此类场景下算法开发与优化的效率。
因此，在 SIMD 与 SIMT 混合编程中，SIMT **不用于替代 SIMD** 处理连续、密集的计算任务，而是用于处理算子中的**局部不规则逻辑**。向量计算可根据任务特点在 SIMD 和 SIMT 之间灵活切换：典型方式是在核函数中通过 **SIMT VF** 处理分支判断、索引映射、离散访存等局部片段，再由 SIMD 继续处理向量或矩阵计算，从而**兼顾灵活性和高吞吐**。

## Vector 工作模式的切换

了解了硬件架构后，我们再来学习 Vector核上的工作模式的切换机制。

在混合编程架构中，AI Vector 核的计算单元包括 SIMD 和 SIMT 两种，并以`Vector Function`（VF）为粒度进行两个工作模式间的切换。两种工作模式共享ALU计算单元，但是同一个时刻只有一个VF在执行，硬件可保障VF是顺序执行的。

基于此，Ascend C 抽象出了 VF 的软件概念，用于表示在 SIMT 或 SIMD 硬件计算单元上执行的特定功能代码段。用户通过编写 SIMD Vector Function（SIMD VF）或 SIMT Vector Function（SIMT VF）来调用对应的执行单元完成计算任务，在算子核函数中调用不同类型的 Vector Function，以实现 SIMD/SIMT 硬件单元的切换使用。

如下图所示，`simt_func` 表示在 SIMT 硬件单元上执行的代码段，`simd_func` 表示在 SIMD 硬件单元上执行的代码段，代码段属性通过 `__simt_vf__`、`__simd_vf__` 来标识。核函数通过 `asc_vf_call` 调用对应的代码段。

![](images/03_05_hybrid/hybrid_vector_function.png)

AI Vector 核执行时，Scalar 计算单元会将 Vector Function 发射到 Vector Function Queue 中，后续串行执行每个 Vector Function。**Vector Function 属于 `PIPE_V` 流水**，与负责数据搬运的 MTE 流水（`PIPE_MTE2`/`PIPE_MTE3`）相互独立，因此 **VF 的计算可以与 MTE 的数据搬运并行执行**，从而相互隐藏延迟。

## 混合编程的适用场景

混合编程适用于算子功能既包含 SIMD 擅长的连续规整计算，也包含 SIMT 擅长的离散访问等任务的场景，例如：

- **复杂分支判断的算子**：适合使用 SIMT 完成计算任务，而输入输出的数据搬运较规整连续，适合使用 SIMD 搬运接口完成数据拷贝。
- **离散访问场景**：适合使用 SIMT 完成计算任务，当所需访问的数据量远小于 Unified Buffer 可用空间时，可以使用 SIMD 搬运接口完成数据拷贝，使 SIMT 编程能够直接从 Unified Buffer 读取数据，提高内存访问效率。
- **大 / 小 Shape 分化场景**：大 Shape 场景用 SIMD 并行计算效率更高，小 Shape 不规整场景适合用 SIMT 完成计算，可区分场景分别实现以泛化整体性能。

## 术语速查

<table>
  <thead>
    <tr>
      <th>术语</th>
      <th>说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>混合编程</td>
      <td>在同一个核函数中，通过调用 SIMD VF 和 SIMT VF，让 SIMD 与 SIMT 硬件单元协同完成计算的编程方式</td>
    </tr>
    <tr>
      <td>SIMD 为主、SIMT 为辅</td>
      <td>混合编程的设计原则：SIMD 承担绝大部分连续、密集的计算算力，SIMT 只负责局部不规则逻辑</td>
    </tr>
    <tr>
      <td>不规则场景</td>
      <td>复杂分支判断、离散访存等难以用 SIMD 批量处理，但适合 SIMT 逐线程处理的场景</td>
    </tr>
    <tr>
      <td>Vector Function（VF）</td>
      <td>表示在 SIMT 或 SIMD 硬件计算单元上执行的特定功能代码段的软件概念</td>
    </tr>
  </tbody>
</table>

## 小节小结

本小节介绍了 SIMD 与 SIMT 混合编程模型的设计背景：

- **设计动机**：为提升向量计算在复杂控制流、离散访存场景下的编程灵活性，AI Vector 核在 SIMD 基础上新增了 SIMT 硬件单元。
- **算力分工**：矩阵计算单元和 SIMD 计算单元提供超过 90% 的算力，负责连续、密集的计算；SIMT 作为灵活性补充，专门处理局部不规则逻辑。
- **适用场景**：复杂分支判断的算子、离散访问场景、大/小 Shape 分化场景都适合使用混合编程。

下一小节将学习混合编程中编程界面，包括核函数与 VF 函数的调用层级，了解如何在代码层面组织 SIMD 和 SIMT 的协同调用。

## 课后练习

本节介绍了 SIMD 与 SIMT 混合编程的设计动机和适用场景，请根据学习内容完成以下题目进行自测。

1. （判断题）在 SIMD 与 SIMT 混合编程架构中，SIMT 用于替代 SIMD 处理连续、密集的计算任务，是算力的主要来源。

2. （单选题）在 SIMD 与 SIMT 混合编程模型中，矩阵计算单元和 SIMD 计算单元共同提供的算力占比大约是多少？  
    A. 50%  
    B. 70%  
    C. 超过 90%  
    D. 100%  

3. （多选题）以下哪些场景适合使用 SIMD 与 SIMT 混合编程？  
    A. 输入输出连续、但存在复杂分支判断的算子  
    B. 访问数据量远小于 Unified Buffer 可用空间的离散访问场景  
    C. 大 Shape 且访存规整的密集矩阵乘法  
    D. 大/小 Shape 分化，需要分别处理的场景  

**执行以下代码获取答案。**

In [ ]:
!cat answer/03.05.01_answer.txt
